In [141]:
import pandas as pd
import numpy as np
import re

# Định nghĩa các ký tự tiếng Việt dùng cho Regex
viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
)

def remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    return text.strip()

In [142]:
# Chọn 1 bảng bất kỳ để làm Test Data
test_name = "Energy_Consumption"

try:
    df_raw_test = pd.read_csv(f"{test_name}.csv")
    print(f" Đọc thành công file Test: {test_name}.csv | Kích thước gốc: {df_raw_test.shape}")
    display(df_raw_test.head(3))
except FileNotFoundError:
    print(f" Không tìm thấy file {test_name}.csv. Vui lòng kiểm tra lại tên file.")

 Đọc thành công file Test: Energy_Consumption.csv | Kích thước gốc: (65, 22)


,Energy Consumption,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,...,Nov,Dec,YTD,Q1,Q2,Q3,Q4,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,Coal/ Than 5A (DV),Ton,1233.522,983.00,2414.661,2198.795,2197.301,2170.990,2442.916,2148.882,...,1649.300,2061.215,NaN,4631.183,6567.086,6310.711,5464.470,NaN,NaN,NaN
1,Coal/ Than 4A.1 (DV),Ton,443.000,251.25,543.069,88.335,774.190,319.418,508.880,630.609,...,113.760,369.092,NaN,1237.319,1181.943,1584.048,501.752,NaN,NaN,NaN
2,Coal/ Than import coal (DV) (than nhập khẩu),Ton,0.000,92.48,430.117,722.734,0.000,434.630,375.270,77.210,...,352.255,533.023,NaN,522.597,1157.364,452.480,1436.158,NaN,NaN,NaN


## Columns Description

---

### 1. Thông tin chung
* **`name`**: Tên sản phẩm. Bao gồm tên tiếng Anh và tiếng Việt (Ví dụ: Coal/ Than 5A).
* **`unit`**: Đơn vị tính. Ở đây là `Ton` (Tấn).
* **`name_no_vn`**: Tên sản phẩm đã lược bỏ tiếng Việt, chỉ giữ lại tiếng Anh hoặc mã hiệu để thuận tiện cho việc xử lý dữ liệu (Data processing).

### 2. Dữ liệu thời gian (Tháng)
Các cột từ `jan` đến `dec` đại diện cho khối lượng tiêu thụ/nhập kho tương ứng với 12 tháng trong năm:
| Cột | Tháng |
| :--- | :--- |
| **jan** | Tháng 1 |
| **feb** | Tháng 2 |
| **mar** | Tháng 3 |
| **apr** | Tháng 4 |
| **may** | Tháng 5 |
| **jun** | Tháng 6 |
| **jul** | Tháng 7 |
| **aug** | Tháng 8 |
| **sep** | Tháng 9 |
| **oct** | Tháng 10 |
| **nov** | Tháng 11 |
| **dec** | Tháng 12 |

### 3. Các ký hiệu đặc biệt trong dữ liệu
* **Số thập phân**: Sử dụng dấu chấm `.` để phân tách (Ví dụ: `1233.522` tấn).
* **`0.000`**: Giá trị bằng không (không có phát sinh trong tháng).
* **`NaN`**: (Not a Number) Dữ liệu trống hoặc chưa có thông tin ghi nhận.
* **Ký hiệu (DV), (YB)**: Đại diện cho các **Cơ sở phát thải (Emission Facilities)** hoặc chi nhánh. ESG yêu cầu báo cáo chi tiết theo từng địa điểm vận hành.

##BƯỚC 1 - Chuẩn hóa Header và Lọc Cột

In [143]:
def clean_headers_and_columns(df):
    df_temp = df.copy()
    if df_temp.iloc[0].astype(str).str.contains('unit', case=False, na=False).any():
        df_temp.columns = [str(c).strip() for c in df_temp.iloc[0]]
        df_temp = df_temp[1:].reset_index(drop=True)

    valid_cols = [c for c in df_temp.columns if not str(c).lower().startswith('unnamed') and str(c).lower() != 'nan']
    df_temp = df_temp[valid_cols]

    cols = list(df_temp.columns)
    if len(cols) >= 2:
        cols[0], cols[1] = "original_name", "unit"
        df_temp.columns = cols
    return df_temp

# --- CHẠY TEST BƯỚC 1 ---
df_step1 = clean_headers_and_columns(df_raw_test)
print(f"--- KẾT QUẢ BƯỚC 1: Kích thước: {df_step1.shape} ---")
display(df_step1.head(10))

--- KẾT QUẢ BƯỚC 1: Kích thước: (65, 19) ---


,original_name,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,Q1,Q2,Q3,Q4
0,Coal/ Than 5A (DV),Ton,1233.522,983.00,2414.661,2198.795,2197.301,2170.990,2442.916,2148.882,1718.913,1753.955,1649.300,2061.215000,NaN,4631.183,6567.086,6310.711,5464.470000
1,Coal/ Than 4A.1 (DV),Ton,443.000,251.25,543.069,88.335,774.190,319.418,508.880,630.609,444.559,18.900,113.760,369.092000,NaN,1237.319,1181.943,1584.048,501.752000
2,Coal/ Than import coal (DV) (than nhập khẩu),Ton,0.000,92.48,430.117,722.734,0.000,434.630,375.270,77.210,0.000,550.880,352.255,533.023000,NaN,522.597,1157.364,452.480,1436.158000
3,Than cám qua sàng / Than 5A under screen DV,Ton,114.000,88.00,206.339,209.205,204.699,201.010,188.848,204.034,133.775,153.045,156.700,175.785000,NaN,408.339,614.914,526.657,485.530000
4,Coal/ Than 5A (YB),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
5,Coal/ Than 4A.1 (YB),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.623114,NaN,0.000,0.000,0.000,27.623114
6,Coal/ Than import coal (YB) (than nhập khẩu),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
7,Than cám qua sàng / Than 5A under screen YB,Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
8,Coal/ Than 5A (DV+YB),Ton,1233.522,983.00,2414.661,2198.795,2197.301,2170.990,2442.916,2148.882,1718.913,1753.955,1649.300,2061.215000,0.0,4631.183,6567.086,6310.711,5464.470000
9,Coal/ Than 4A.1 (DV+YB),Ton,443.000,251.25,543.069,88.335,774.190,319.418,508.880,630.609,444.559,18.900,113.760,396.715114,0.0,1237.319,1181.943,1584.048,529.375114


##BƯỚC 2 - Lọc Rác Theo Dòng

In [144]:
def filter_garbage_rows(df):
    df_temp = df.copy()
    df_temp = df_temp.dropna(subset=['original_name'])
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip() != '']
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip().str.lower() != 'nan']
    return df_temp

# --- CHẠY TEST BƯỚC 2 ---
df_step2 = filter_garbage_rows(df_step1)
print(f"--- KẾT QUẢ BƯỚC 2: Kích thước: {df_step2.shape} ---")
display(df_step2.head(10))

--- KẾT QUẢ BƯỚC 2: Kích thước: (64, 19) ---


,original_name,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,YTD,Q1,Q2,Q3,Q4
0,Coal/ Than 5A (DV),Ton,1233.522,983.00,2414.661,2198.795,2197.301,2170.990,2442.916,2148.882,1718.913,1753.955,1649.300,2061.215000,NaN,4631.183,6567.086,6310.711,5464.470000
1,Coal/ Than 4A.1 (DV),Ton,443.000,251.25,543.069,88.335,774.190,319.418,508.880,630.609,444.559,18.900,113.760,369.092000,NaN,1237.319,1181.943,1584.048,501.752000
2,Coal/ Than import coal (DV) (than nhập khẩu),Ton,0.000,92.48,430.117,722.734,0.000,434.630,375.270,77.210,0.000,550.880,352.255,533.023000,NaN,522.597,1157.364,452.480,1436.158000
3,Than cám qua sàng / Than 5A under screen DV,Ton,114.000,88.00,206.339,209.205,204.699,201.010,188.848,204.034,133.775,153.045,156.700,175.785000,NaN,408.339,614.914,526.657,485.530000
4,Coal/ Than 5A (YB),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
5,Coal/ Than 4A.1 (YB),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.623114,NaN,0.000,0.000,0.000,27.623114
6,Coal/ Than import coal (YB) (than nhập khẩu),Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
7,Than cám qua sàng / Than 5A under screen YB,Ton,0.000,0.00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000000
8,Coal/ Than 5A (DV+YB),Ton,1233.522,983.00,2414.661,2198.795,2197.301,2170.990,2442.916,2148.882,1718.913,1753.955,1649.300,2061.215000,0.0,4631.183,6567.086,6310.711,5464.470000
9,Coal/ Than 4A.1 (DV+YB),Ton,443.000,251.25,543.069,88.335,774.190,319.418,508.880,630.609,444.559,18.900,113.760,396.715114,0.0,1237.319,1181.943,1584.048,529.375114


##BƯỚC 3 - Chuyển dọc dữ liệu

In [145]:
def melt_data(df, category_name):
    df_temp = df.copy()
    df_temp.columns = df_temp.columns.astype(str).str.strip()
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    valid_months = [m for m in months if m in df_temp.columns]

    if not valid_months:
        return pd.DataFrame() # Trả về DF rỗng nếu không có cột tháng

    df_long = df_temp.melt(id_vars=["original_name", "unit"], value_vars=valid_months,
                           var_name="period", value_name="value")
    df_long.insert(0, "category", category_name)
    return df_long

# --- CHẠY TEST BƯỚC 3 ---
df_step3 = melt_data(df_step2, test_name)
print(f"--- KẾT QUẢ BƯỚC 3 (MELT): Kích thước: {df_step3.shape} ---")
display(df_step3.head(10))

--- KẾT QUẢ BƯỚC 3 (MELT): Kích thước: (768, 5) ---


,category,original_name,unit,period,value
0,Energy_Consumption,Coal/ Than 5A (DV),Ton,Jan,1233.522
1,Energy_Consumption,Coal/ Than 4A.1 (DV),Ton,Jan,443.000
2,Energy_Consumption,Coal/ Than import coal (DV) (than nhập khẩu),Ton,Jan,0.000
3,Energy_Consumption,Than cám qua sàng / Than 5A under screen DV,Ton,Jan,114.000
4,Energy_Consumption,Coal/ Than 5A (YB),Ton,Jan,0.000
5,Energy_Consumption,Coal/ Than 4A.1 (YB),Ton,Jan,0.000
6,Energy_Consumption,Coal/ Than import coal (YB) (than nhập khẩu),Ton,Jan,0.000
7,Energy_Consumption,Than cám qua sàng / Than 5A under screen YB,Ton,Jan,0.000
8,Energy_Consumption,Coal/ Than 5A (DV+YB),Ton,Jan,1233.522
9,Energy_Consumption,Coal/ Than 4A.1 (DV+YB),Ton,Jan,443.000


##BƯỚC 4 - Chuẩn hóa tên (Xóa tiếng Việt)

In [146]:
def normalize_names(df_long):
    df_temp = df_long.copy()
    df_temp.insert(1, "name", df_temp["original_name"].apply(remove_vietnamese))
    rules = {r'Than cám qua sàng / Than ': 'Coal ', r'/ Than non': '', r'/ Than ': ' '}
    for old, new in rules.items():
        df_temp["name"] = df_temp["name"].str.replace(old, new, regex=True)
    df_temp["name"] = df_temp["name"].str.strip()
    return df_temp

# --- CHẠY TEST BƯỚC 4 ---
df_step4 = normalize_names(df_step3)
print("--- KẾT QUẢ BƯỚC 4 ---")
display(df_step4.head(10))

--- KẾT QUẢ BƯỚC 4 ---


,category,name,original_name,unit,period,value
0,Energy_Consumption,Coal 5A (DV),Coal/ Than 5A (DV),Ton,Jan,1233.522
1,Energy_Consumption,Coal 4A.1 (DV),Coal/ Than 4A.1 (DV),Ton,Jan,443.000
2,Energy_Consumption,Coal import coal (DV),Coal/ Than import coal (DV) (than nhập khẩu),Ton,Jan,0.000
3,Energy_Consumption,Coal 5A under screen DV,Than cám qua sàng / Than 5A under screen DV,Ton,Jan,114.000
4,Energy_Consumption,Coal 5A (YB),Coal/ Than 5A (YB),Ton,Jan,0.000
5,Energy_Consumption,Coal 4A.1 (YB),Coal/ Than 4A.1 (YB),Ton,Jan,0.000
6,Energy_Consumption,Coal import coal (YB),Coal/ Than import coal (YB) (than nhập khẩu),Ton,Jan,0.000
7,Energy_Consumption,Coal 5A under screen YB,Than cám qua sàng / Than 5A under screen YB,Ton,Jan,0.000
8,Energy_Consumption,Coal 5A (DV+YB),Coal/ Than 5A (DV+YB),Ton,Jan,1233.522
9,Energy_Consumption,Coal 4A.1 (DV+YB),Coal/ Than 4A.1 (DV+YB),Ton,Jan,443.000


##BƯỚC 5 - Xử lý số liệu và Ép kiểu

In [147]:
import numpy as np
import pandas as pd

def process_numeric_data(df_long):
    df_temp = df_long.copy()
    df_temp["unit"] = df_temp["unit"].astype(str).str.strip()

    # 1. Chuyển các giá trị trống/không hợp lệ thành NaN
    df_temp["value"] = df_temp["value"].astype(str).str.strip()
    df_temp["value"] = df_temp["value"].str.replace(',', '', regex=False)
    df_temp["value"] = df_temp["value"].replace(['-', '', 'nan', 'NaN', 'None'], np.nan)
    df_temp["value"] = pd.to_numeric(df_temp["value"], errors="coerce")

    # 2. Tạo sẵn dữ liệu cho cột 'has_data' và 'Month'
    has_data_series = df_temp["value"].notna().astype(int)

    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    month_series = df_temp["period"].map(month_map)

    # 3. ĐIỀN TRUNG VỊ (MEDIAN) CỦA RIÊNG TỪNG SẢN PHẨM TRONG 12 THÁNG
    df_temp["value"] = df_temp.groupby("name")["value"].transform(lambda x: x.fillna(x.median()))

    # Đề phòng trường hợp một sản phẩm trống toàn bộ cả 12 tháng (median cũng là NaN) -> điền bằng 0
    df_temp["value"] = df_temp["value"].fillna(0).round(3)

    # 4. CHÈN CỘT VÀO ĐÚNG VỊ TRÍ
    if "period" in df_temp.columns:
        period_idx = df_temp.columns.get_loc("period")
        df_temp.insert(period_idx + 1, "Month", month_series)

    # Chèn cột 'has_data' (1/0) ngay sau cột 'value'
    if "value" in df_temp.columns:
        value_idx = df_temp.columns.get_loc("value")
        df_temp.insert(value_idx + 1, "has_data", has_data_series)

    return df_temp

# --- CHẠY TEST BƯỚC 5 MỚI ---
df_step5 = process_numeric_data(df_step4)
print(f"--- KẾT QUẢ BƯỚC 5: Kích thước: {df_step5.shape} ---")
display(df_step5.head(10))

--- KẾT QUẢ BƯỚC 5: Kích thước: (768, 8) ---


,category,name,original_name,unit,period,Month,value,has_data
0,Energy_Consumption,Coal 5A (DV),Coal/ Than 5A (DV),Ton,Jan,1,1233.522,1
1,Energy_Consumption,Coal 4A.1 (DV),Coal/ Than 4A.1 (DV),Ton,Jan,1,443.000,1
2,Energy_Consumption,Coal import coal (DV),Coal/ Than import coal (DV) (than nhập khẩu),Ton,Jan,1,0.000,1
3,Energy_Consumption,Coal 5A under screen DV,Than cám qua sàng / Than 5A under screen DV,Ton,Jan,1,114.000,1
4,Energy_Consumption,Coal 5A (YB),Coal/ Than 5A (YB),Ton,Jan,1,0.000,1
5,Energy_Consumption,Coal 4A.1 (YB),Coal/ Than 4A.1 (YB),Ton,Jan,1,0.000,1
6,Energy_Consumption,Coal import coal (YB),Coal/ Than import coal (YB) (than nhập khẩu),Ton,Jan,1,0.000,1
7,Energy_Consumption,Coal 5A under screen YB,Than cám qua sàng / Than 5A under screen YB,Ton,Jan,1,0.000,1
8,Energy_Consumption,Coal 5A (DV+YB),Coal/ Than 5A (DV+YB),Ton,Jan,1,1233.522,1
9,Energy_Consumption,Coal 4A.1 (DV+YB),Coal/ Than 4A.1 (DV+YB),Ton,Jan,1,443.000,1


##Đóng gói Pipeline (Gom các hàm lại)

In [148]:
def clean_table(df_raw, category_name):
    df1 = clean_headers_and_columns(df_raw)
    df2 = filter_garbage_rows(df1)
    df3 = melt_data(df2, category_name)

    if df3.empty:
        return df3

    df4 = normalize_names(df3)
    df5 = process_numeric_data(df4)
    return df5

print("Đã khởi tạo hàm Pipeline tổng: clean_table()")

Đã khởi tạo hàm Pipeline tổng: clean_table()


##Chạy toàn bộ cho các CSV và xuất Master Data

In [149]:
file_names = ['Energy_Consumption', 'Greenhouse_Gas_Emission', 'Energy_Cost']
cleaned_tables = []

print("--- BẮT ĐẦU XỬ LÝ HÀNG LOẠT ---")

for name in file_names:
    try:
        df_raw = pd.read_csv(f"{name}.csv")
        print(f"\nĐang xử lý bảng: {name} (Gốc: {df_raw.shape})")

        # Gọi Pipeline
        df_clean = clean_table(df_raw, name)

        if not df_clean.empty:
            cleaned_tables.append(df_clean)
            print(f"  -> Thành công! Kích thước sạch: {df_clean.shape}")
        else:
            print("  -> Bỏ qua vì không có cột tháng hợp lệ.")

    except FileNotFoundError:
        print(f" Bỏ qua: Không tìm thấy {name}.csv")

# Gộp bảng
if cleaned_tables:
    final_df = pd.concat(cleaned_tables, ignore_index=True)

    # Cập nhật danh sách các cột để bao gồm 'Month' và 'has_data'
    final_cols = ['category', 'name', 'original_name', 'unit', 'period', 'Month', 'value', 'has_data']

    # Kiểm tra xem các cột có thực sự tồn tại trước khi lọc để tránh lỗi
    existing_cols = [col for col in final_cols if col in final_df.columns]
    final_df = final_df[existing_cols]

    # Xuất file CSV
    final_df.to_csv("Energy_GHG_Cleaned.csv", index=False)

    print(f"\n XONG! Đã lưu file Master (Tổng dòng: {len(final_df)}).")
    display(final_df.sample(20))
else:
    print("\n KHÔNG CÓ DỮ LIỆU ĐỂ GỘP! Vui lòng kiểm tra lại file CSV gốc.")

--- BẮT ĐẦU XỬ LÝ HÀNG LOẠT ---

Đang xử lý bảng: Energy_Consumption (Gốc: (65, 22))
  -> Thành công! Kích thước sạch: (768, 8)

Đang xử lý bảng: Greenhouse_Gas_Emission (Gốc: (30, 22))
  -> Thành công! Kích thước sạch: (348, 8)

Đang xử lý bảng: Energy_Cost (Gốc: (13, 22))
  -> Thành công! Kích thước sạch: (156, 8)

 XONG! Đã lưu file Master (Tổng dòng: 1272).


/tmp/ipykernel_2769/880152919.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_temp["value"] = df_temp["value"].replace(['-', '', 'nan', 'NaN', 'None'], np.nan)


,category,name,original_name,unit,period,Month,value,has_data
432,Energy_Consumption,Total Electrical Consumption,Total Electrical Consumption,GJ,Jul,7,8405.401,1
965,Greenhouse_Gas_Emission,Specific GHG Scope1-Coal Gas (DV),Specific GHG Scope1-Coal Gas (DV),kgCO2/ton,Jul,7,673.570,1
544,Energy_Consumption,Solid Biomass (YB),Solid Biomass (YB) (Củi+rác đốt),Ton,Sep,9,0.000,1
593,Energy_Consumption,Coal heating value import coal,Coal heating value import coal (Nhiệt trị theo...,cal/g,Oct,10,5520.000,1
1073,Greenhouse_Gas_Emission,Total GHG Scope1-Coal Gas (YB),Total GHG Scope1-Coal Gas (YB),Ton CO2,Nov,11,0.000,1
879,Greenhouse_Gas_Emission,Specific GHG Scope1-Coal Gas (YB),Specific GHG Scope1-Coal Gas (YB),kgCO2/ton,Apr,4,0.000,0
21,Energy_Consumption,Coal Gas (YB),Coal Gas (YB) (Nhiệt trị theo phiếu kiểm tra),Mcal,Jan,1,0.000,0
26,Energy_Consumption,LPG for Process used (Station),LPG for Process used (Station)/ LPG dùng cho t...,Ton,Jan,1,0.000,0
455,Energy_Consumption,Coal 5A under screen YB,Than cám qua sàng / Than 5A under screen YB,Ton,Aug,8,0.000,0
754,Energy_Consumption,Total Renewable Energy (YB),Total Renewable Energy (YB),GJ,Dec,12,0.000,1
